# Masking Model — Registration

Logs and registers `MaskingModel` in Unity Catalog with MLflow.

## Prerequisites
- Databricks Runtime ML 14.x or later
- Unity Catalog enabled
- Model code synced to workspace under `../model/`

## Next Steps
1. Run `deploy_endpoint.py` with `MODEL_NAME = "doc_masking"` and the version printed below
2. Set `DATABRICKS_MASKING_ENDPOINT` on the FastAPI backend

In [ ]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [ ]:
# ---------------------------------------------------------------------------
# Configuration — update before running
# ---------------------------------------------------------------------------
CATALOG    = "datascience_dev_bronze_sandbox"
SCHEMA     = "ds_document_deidentification"
MODEL_NAME = "doc_masking"

UC_MODEL_PATH  = f"{CATALOG}.{SCHEMA}.{MODEL_NAME}"
UC_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/sessions"

print(f"Model : {UC_MODEL_PATH}")
print(f"Volume: {UC_VOLUME_PATH}")

In [ ]:
import json
import os
import sys

import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# Add repo root to path so relative imports inside model/ work
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
module_dir = os.path.join("/Workspace" + os.path.dirname(notebook_path), "../../..")
sys.path.insert(0, module_dir)
print(f"Module dir: {module_dir}")

In [ ]:
from privasee.databricks.model.masking_model import MaskingModel
print("MaskingModel imported successfully")

In [ ]:
# ---------------------------------------------------------------------------
# MLflow signature
# ---------------------------------------------------------------------------
input_schema = Schema([
    ColSpec("string", "session_id"),
    ColSpec("string", "entities_to_mask"),  # JSON string: list of entity dicts
])

output_schema = Schema([
    ColSpec("string", "session_id"),
    ColSpec("string", "status"),
    ColSpec("long",   "entities_masked"),
    ColSpec("string", "error_message", required=False),
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
print("Signature defined")

In [ ]:
# ---------------------------------------------------------------------------
# Input example
# ---------------------------------------------------------------------------
input_example = pd.DataFrame({
    "session_id": ["example_session_123"],
    "entities_to_mask": [json.dumps([
        {
            "id": "e1",
            "entity_type": "Full Name",
            "original_text": "John Smith",
            "replacement_text": "Jane Doe",
            "bounding_box": [0.05, 0.08, 0.2, 0.025],
            "strategy": "Fake Data",
            "approved": True,
            "page_number": 1,
        }
    ])],
})
print("Input example created")

In [ ]:
# ---------------------------------------------------------------------------
# Log model to MLflow
# ---------------------------------------------------------------------------
mlflow.set_registry_uri("databricks-uc")

username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
mlflow.set_experiment(f"/Users/{username}/privasee_experiments")

requirements_path = os.path.join(os.path.dirname(module_dir), "databricks/requirements.txt")

with mlflow.start_run(run_name="privasee_masking_model_registration") as run:
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=MaskingModel(),
        signature=signature,
        input_example=input_example,
        pip_requirements=requirements_path,
        code_paths=[os.path.join(module_dir, "privasee")],
    )
    mlflow.log_param("catalog", CATALOG)
    mlflow.log_param("schema", SCHEMA)
    mlflow.log_param("model_name", MODEL_NAME)
    mlflow.log_param("uc_volume_path", UC_VOLUME_PATH)

print(f"Model logged: {model_info.model_uri}")

In [ ]:
# ---------------------------------------------------------------------------
# Register in Unity Catalog
# ---------------------------------------------------------------------------
registered_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=UC_MODEL_PATH,
    tags={"project": "privasee", "type": "masking"},
)

MODEL_VERSION = str(registered_model.version)
print(f"Registered: {UC_MODEL_PATH} v{MODEL_VERSION}")

In [ ]:
print(f"""
Model Path:    {UC_MODEL_PATH}
Model Version: {MODEL_VERSION}

Next: run deploy_endpoint.py with:
  MODEL_NAME    = \"{MODEL_NAME}\"
  MODEL_VERSION = \"{MODEL_VERSION}\"

Then set on FastAPI backend:
  DATABRICKS_MASKING_ENDPOINT = https://<host>/serving-endpoints/<endpoint-name>/invocations
""")